# Crop utterances from zoom_recording.wav

Locates each utterance in the full Zoom recording via cross-correlation with the
existing crops in `orig_wavs/`, then re-saves the stereo 44100 Hz crops and writes
transcript txt files to `inputs/`.

Note: resampling to 16 kHz mono was done separately after running this notebook.

In [ ]:
import os
import numpy as np
import scipy.signal
import torchaudio

INPUTS_DIR = os.path.abspath('.')
ORIG_DIR   = os.path.join(INPUTS_DIR, 'orig_wavs')
FULL_WAV   = os.path.join(ORIG_DIR, 'zoom_recording.wav')

In [ ]:
TRANSCRIPTS = {
    'george_party_1':   "I'm not going to George's party.",
    'george_party_2':   "I'm not going to George's party.",
    'george_party_3':   "I'm not going to George's party.",
    'green_ball_1':     'Pick up the green ball.',
    'green_ball_2':     'Pick up the green ball.',
    'janet_broccoli_1': 'Janet likes broccoli?',
    'janet_broccoli_2': 'Janet likes broccoli.',
    'mary_languages_1': 'Mary knows many languages, you know.',
    'mary_languages_2': 'Mary knows many languages you know.',
}

In [ ]:
# Load full recording and convert to mono for cross-correlation
full_wav, sr = torchaudio.load(FULL_WAV)
print(f'Full recording: {sr} Hz, {full_wav.shape[0]} ch, {full_wav.shape[1]/sr:.2f}s')

full_mono = full_wav.mean(dim=0).numpy()  # (T,)

In [ ]:
def find_clip(clip_np, full_np, sr):
    """Return (start_sec, end_sec, start_sample) of clip inside full via cross-correlation."""
    corr    = scipy.signal.correlate(full_np, clip_np, mode='valid')
    lag     = int(np.argmax(np.abs(corr)))
    return lag / sr, (lag + len(clip_np)) / sr, lag, corr[lag]

In [ ]:
# Locate each clip in the full recording
timestamps = {}  # utt_id -> (start_s, end_s, start_sample)

for utt_id in TRANSCRIPTS:
    clip_wav, _ = torchaudio.load(os.path.join(ORIG_DIR, f'{utt_id}.wav'))
    clip_mono   = clip_wav.mean(dim=0).numpy()

    start_s, end_s, lag, peak = find_clip(clip_mono, full_mono, sr)
    timestamps[utt_id] = (start_s, end_s, lag)
    print(f'{utt_id:25s}  {start_s:6.2f}s - {end_s:6.2f}s  (peak corr {peak:.4f})')

In [ ]:
# Save stereo 44100 Hz crops to orig_wavs/${utt_id}.wav
for utt_id, (start_s, end_s, lag) in timestamps.items():
    clip_len     = int(round((end_s - start_s) * sr))
    crop         = full_wav[:, lag:lag + clip_len]  # [2, T]
    out_path     = os.path.join(ORIG_DIR, f'{utt_id}.wav')
    torchaudio.save(out_path, crop, sr)
    print(f'Saved {out_path}  ({crop.shape[1]/sr:.2f}s)')

In [ ]:
# Write transcript txt files to inputs/${utt_id}.txt
for utt_id, transcript in TRANSCRIPTS.items():
    out_path = os.path.join(INPUTS_DIR, f'{utt_id}.txt')
    with open(out_path, 'w') as f:
        f.write(transcript)
    print(f'Wrote {out_path}')